# Phase 5: a bounded 64,000-query training run

The hosted T4 run passed on 24 October 2026: 2,000 updates and 81 tests.
See the [reviewed results](https://github.com/WeaversMask/temporal-relational-gnn/blob/main/reports/TRAINING_RUN_STATUS.md).

The first hosted training pilot passed. This next run starts a **fresh model** using the same
constant node features, relative time, architecture and reference training schedule. It selects
64,000 official training queries for 2,000 updates and evaluates a fixed 8,192-query validation
subset. The activity-count logistic-regression baseline fits the same selected training rows and
uses the same validation rows. The test split stays untouched.

This is a bounded supervised training run, not the final predictor or an inference-speed benchmark.
It records validation before and after training and checks trained-checkpoint equivalence on one
identical sampled validation batch across reference, HGAM and blocks. A later measurement will
compare trained inference speed using a fixed checkpoint and paired sampled occurrences.

1. Save a copy to Drive. Runtime storage remains temporary; this notebook does not mount Drive.
2. In Colab's managed sessions, disconnect other runtimes before this run. Use one active runtime
   so an observed start/end account balance difference can be attributed to this runtime interval.
3. Select **T4 GPU**, save, and connect. For CPU development, select a CPU runtime and set
   `DEVICE = "cpu"` below.
4. Enable the existing read-only `GITHUB_TOKEN` in Secrets. Fill the actual unit/hour rate and
   remaining balance when Colab displays them. Leave an unavailable observation `None`.
5. Run cells in order, keeping the frozen configuration. If any step fails, run the ending-balance
   and ZIP-export cells. Export before disconnecting or deleting the runtime.

The archive includes **the resumable checkpoint**, reports, configuration and logs; it excludes the
database. Keep the ZIP. To resume this run, extract your own earlier `checkpoints/training.pt`,
set `RESUME = True`, and use the restore cell below. Do not restore the smaller notebook 04 pilot:
its 3,200-row cohort is different. Resuming completes the same total 2,000-update target.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
# Set True after checking that other managed runtimes are disconnected.
ISOLATED_RUNTIME_CONFIRMED = False
RESUME = False
RESUME_CHECKPOINT = None  # Existing own training.pt path, or None to upload when resuming.
RESUME_RUN_ID = None  # On a fresh VM, optionally retain the ID from your earlier exported ZIP.
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"  # For resume, use the source commit recorded in the earlier checkout.log.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Preserve the same run folder if this cell is rerun in this notebook session.
if "_TRAINING_RUN_DIR" not in globals():
    if RESUME_RUN_ID is not None:
        if not RESUME or not isinstance(RESUME_RUN_ID, str) or not RESUME_RUN_ID:
            raise ValueError("RESUME_RUN_ID needs RESUME=True and the earlier run ID")
        if any(character not in "0123456789TZ-abcdef" for character in RESUME_RUN_ID):
            raise ValueError("Use only the run ID from the earlier exported ZIP filename")
    _TRAINING_RUN_ID = RESUME_RUN_ID or (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
    )
    _TRAINING_RUN_DIR = Path("/content/temporary-trgnn-training-run/runs") / _TRAINING_RUN_ID
    _TRAINING_RUN_DIR.mkdir(parents=True, exist_ok=False)
run_id = _TRAINING_RUN_ID
RUN_DIR = _TRAINING_RUN_DIR
CHECKPOINT_PATH = RUN_DIR / "checkpoints/training.pt"
print(f"Run directory: {RUN_DIR}", flush=True)
print(f"Checkpoint: {CHECKPOINT_PATH}", flush=True)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


def observed_nonnegative_number(name, value):
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")


observed_nonnegative_number("COMPUTE_UNITS_PER_HOUR", COMPUTE_UNITS_PER_HOUR)
observed_nonnegative_number("COMPUTE_UNITS_REMAINING", COMPUTE_UNITS_REMAINING)
budget_path = RUN_DIR / "colab-budget.json"
if not budget_path.exists():
    budget_path.write_text(
        json.dumps(
            {
                "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
                "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
                "units_remaining_observed": COMPUTE_UNITS_REMAINING,
                "isolated_runtime_confirmed": ISOLATED_RUNTIME_CONFIRMED,
                "planned_total_units": 200,
                "reserve_units": RESERVE_UNITS,
                "scope": "current_runtime_interval_only",
                "resume_requested": RESUME,
            },
            indent=2,
            allow_nan=False,
        )
        + "\n"
    )
else:
    print("Preserving the original starting budget observation for this run folder.")
metadata_path = RUN_DIR / "notebook-run.json"
if metadata_path.exists():
    run_metadata = json.loads(metadata_path.read_text())
    if run_metadata["device"] != DEVICE:
        raise ValueError("Keep the same device for this run folder and checkpoint")
else:
    run_metadata = {
        "notebook": "05_training_run",
        "run_id": run_id,
        "device": DEVICE,
        "project_ref": PROJECT_REF,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
# Pin reruns of this run folder to the commit used by the first successful checkout.
EFFECTIVE_PROJECT_REF = run_metadata.get("source_commit", PROJECT_REF)
print("Project revision for this run:", EFFECTIVE_PROJECT_REF)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Unavailable budget observations are recorded as null.")

Clone or update the project and install dependencies. Logs and exportable storage already exist.
If setup fails, skip to the ending-balance and export cells.

In [ ]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            'print("x-access-token" if "username" in sys.argv[1].lower() '
            'else os.environ["TRGNN_CLONE_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        clone_environment = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "TRGNN_CLONE_TOKEN": token,
        }
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    if "clone_environment" in globals():
        clone_environment.pop("TRGNN_CLONE_TOKEN", None)

run_logged(["git", "checkout", EFFECTIVE_PROJECT_REF], "checkout.log")
if EFFECTIVE_PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip()
if run_metadata.get("source_commit", source_commit) != source_commit:
    raise RuntimeError("This run folder must retain its original source commit")
run_metadata["source_commit"] = source_commit
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the binary sampler and correctness suite before loading training data. These commands run
in fresh Python processes after installation. A failed check stops this run.

In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

The configuration is frozen in `configs/training_run.json`: **2,000 total updates, batch size 32,
64,000 selected training queries, 8,192 validation queries, 64 channels, fanouts `[16, 8]`,
learning rate 0.001 and seed 42**. Progress reports and checkpoints are written every 100 updates
and at the final update. Each selected training query appears once in this fixed 2,000-update
schedule. The baseline fits those same 64,000 training rows.

The two-layer model trains using the reference schedule. Evaluation compares all three execution
modes on one identical sampled validation batch after training. This validation subset is larger
than the first pilot but is not the full-validation metric. No test labels or raw mutable/text
fields are used.

The graph uses the hosted snapshot with record-time filtering. Historical deletions, merges and
relationship changes have not been reconstructed; those assumptions remain in the audit.

In [ ]:
import yaml

TRAINING_CONFIG = json.loads((PROJECT_ROOT / "configs/training_run.json").read_text())
frozen_config_path = RUN_DIR / "frozen-training-config.json"
if frozen_config_path.exists():
    if json.loads(frozen_config_path.read_text()) != TRAINING_CONFIG:
        raise RuntimeError("This run folder must retain its original frozen configuration")
else:
    frozen_config_path.write_text(json.dumps(TRAINING_CONFIG, indent=2) + "\n")
print(json.dumps(TRAINING_CONFIG, indent=2))

pins = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())
source_receipt = DATA_DIR / ".trgnn-source.json"
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None
required_inputs = [
    DATA_DIR / "manifest.yaml",
    DATA_DIR / "tasks/user-badge/manifest.yaml",
    DATA_DIR / "tasks/user-badge/train.parquet",
    DATA_DIR / "tasks/user-badge/val.parquet",
]
if (DATA_DIR / "manifest.yaml").is_file():
    database_spec = yaml.safe_load((DATA_DIR / "manifest.yaml").read_text())
    required_inputs.extend(DATA_DIR / "db" / f"{name}.parquet" for name in database_spec["tables"])
NEEDS_DOWNLOAD = cached_source != pins["dataset"] or any(not p.is_file() for p in required_inputs)
print("Pinned training/validation inputs need download:", NEEDS_DOWNLOAD)

### Restore this training run, only when resuming

Leave `RESUME = False` for the first run: this creates a fresh model, not a continuation of
notebook 04. After a stop in this notebook session, set it to `True` in the first code cell and
rerun that cell; the current run ID, starting budget observation and frozen configuration are
preserved. If its `training.pt` exists it is reused.

On a fresh VM, set `PROJECT_REF` to the original commit recorded in the earlier `checkout.log`
and retain matching dependency versions. Optionally set `RESUME_RUN_ID` to the ID from the earlier
ZIP filename. Set `RESUME_CHECKPOINT` to your restored checkpoint path, or leave it `None` and
choose the single `training.pt` extracted from your earlier exported ZIP when prompted below.
Keep the earlier archive as well: a fresh VM records a new budget interval and does not reconstruct
missing earlier observations or validation-before results. The checkpoint is copied into this
run's checkpoint folder. Training verifies code, runtime, configuration, selected rows, seed and
saved state before resuming.

In [ ]:
if RESUME:
    if RESUME_CHECKPOINT is None and CHECKPOINT_PATH.is_file():
        RESUME_CHECKPOINT = CHECKPOINT_PATH
    if RESUME_CHECKPOINT is None:
        from google.colab import files

        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(
                "Choose the single training.pt checkpoint from your earlier exported ZIP"
            )
        _, checkpoint_bytes = next(iter(uploaded.items()))
        restored = Path("/content/trgnn-resume") / run_id / "training.pt"
        restored.parent.mkdir(parents=True, exist_ok=True)
        restored.write_bytes(checkpoint_bytes)
        RESUME_CHECKPOINT = restored
        del checkpoint_bytes, uploaded
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)
    if not RESUME_CHECKPOINT.is_file():
        raise FileNotFoundError(f"No previous training checkpoint at {RESUME_CHECKPOINT}")
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if RESUME_CHECKPOINT.resolve() != CHECKPOINT_PATH.resolve():
        shutil.copy2(RESUME_CHECKPOINT, CHECKPOINT_PATH)
    print("Resume checkpoint:", CHECKPOINT_PATH)
else:
    if CHECKPOINT_PATH.exists():
        raise RuntimeError(
            "A checkpoint already exists. Set RESUME=True to continue this training run."
        )
    print("Starting a fresh 64,000-query training run; no checkpoint restored.")

Run the fixed training schedule. Progress, raw losses, training times and completed update counts
are saved every 100 updates and at completion. Checkpoints preserve the model, optimizer,
configuration, query provenance and random state; deterministic step seeds support resuming the
same schedule. If interrupted between checkpoints, resume from the latest saved checkpoint.

Validation is compared before and after the updates, alongside the activity-count baseline fitted
only on the selected training queries. After training, reference, HGAM and blocks must agree on
logits and loss for one validation batch using the same checkpoint and sampled occurrences. This
checks trained-checkpoint equivalence; it does not time inference acceleration. If anything fails,
export the available checkpoint and logs.

In [ ]:
arguments = [
    sys.executable,
    "-m",
    "trgnn.training_pilot",
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--steps",
    str(TRAINING_CONFIG["steps"]),
    "--batch-size",
    str(TRAINING_CONFIG["batch_size"]),
    "--train-queries",
    str(TRAINING_CONFIG["train_queries"]),
    "--val-queries",
    str(TRAINING_CONFIG["val_queries"]),
    "--channels",
    str(TRAINING_CONFIG["channels"]),
    "--fanouts",
    *map(str, TRAINING_CONFIG["fanouts"]),
    "--learning-rate",
    str(TRAINING_CONFIG["learning_rate"]),
    "--seed",
    str(TRAINING_CONFIG["seed"]),
    "--checkpoint-every",
    str(TRAINING_CONFIG["checkpoint_every"]),
    "--report-every",
    str(TRAINING_CONFIG["report_every"]),
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--output",
    str(RUN_DIR / "training-run.json"),
]
if NEEDS_DOWNLOAD:
    arguments.append("--download")
if RESUME:
    arguments.append("--resume")
run_logged(arguments, "training-run.log")

Inspect the run status, completed updates, subset validation metrics and trained-mode logit errors.
A failed or partial attempt still provides raw evidence and may include a resumable checkpoint.
A same-folder resume preserves the preceding JSON report as a separate artifact in the ZIP.
Historical feature availability, final predictive evaluation and trained inference measurements
remain open.

In [ ]:
report_path = RUN_DIR / "training-run.json"
if report_path.is_file():
    report = json.loads(report_path.read_text())
    print("Status:", report.get("status"))
    print("Scope:", report.get("scope"))
    print("Trained:", report.get("trained", False))
    print("Completed steps:", report.get("training", {}).get("completed_steps", 0))
    print("Checkpoint:", report.get("checkpoint", {}))
    for phase in ["validation_before", "validation_after"]:
        print(phase, report.get(phase, {}).get("metrics", {}))
    baseline = report.get("activity_baseline", {})
    print("Activity baseline status:", baseline.get("status"), baseline.get("prediction_source"))
    print(
        "Activity baseline metrics:",
        baseline.get("validation_metrics", baseline.get("metrics", {})),
    )
    equivalence = report.get("trained_mode_equivalence", {})
    print("Trained mode max logit errors:", equivalence.get("max_absolute_logit_error", {}))
    print(
        "Feature audit training ready:",
        report.get("feature_audit", {}).get("training_ready", False),
    )
    if "error" in report:
        print("Error:", report["error"])
    print("Use the saved JSON for raw step history, query rows and full provenance.")
else:
    print("No training report yet; export the available setup logs and checkpoint.")

### Record the ending balance

After training stops, copy the actual remaining units from Colab's resource panel below. Leave
`None` if unavailable. The report records the account balance change between the two observations,
rounded to Colab's two displayed decimal places. It attributes that change to this runtime interval
only if other runtime usage was checked at the start and no balance increase occurred.

The starting observation stays intact when resuming in this session. A fresh-VM resume starts a
new interval, so retain all earlier archives to review total consumption. Previously reported
balances are prior observations, not a preset for this run. Budget observations are independent of
whether training passed; a missing observation does not require repeating the run.

In [ ]:
ENDING_UNITS_REMAINING = None
observed_nonnegative_number("ENDING_UNITS_REMAINING", ENDING_UNITS_REMAINING)
starting_budget = json.loads((RUN_DIR / "colab-budget.json").read_text())
starting_balance = starting_budget["units_remaining_observed"]
account_delta = (
    round(starting_balance - ENDING_UNITS_REMAINING, 2)
    if starting_balance is not None and ENDING_UNITS_REMAINING is not None
    else None
)
isolated_delta = (
    account_delta
    if starting_budget.get("isolated_runtime_confirmed")
    and account_delta is not None
    and account_delta >= 0
    else None
)
ending_budget = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "starting_observation_at_utc": starting_budget.get("recorded_at_utc"),
    "starting_units_remaining_observed": starting_balance,
    "ending_units_remaining_observed": ENDING_UNITS_REMAINING,
    "account_balance_decrease_units": account_delta,
    "balance_delta_precision_decimal_places": 2,
    "isolated_run_consumption_units": isolated_delta,
    "isolated_runtime_confirmed": starting_budget.get("isolated_runtime_confirmed", False),
    "scope": "current_runtime_interval_only",
    "note": (
        "Attribution needs observed balances and an isolated runtime; "
        "earlier intervals are separate."
    ),
}
(RUN_DIR / "budget-end.json").write_text(
    json.dumps(ending_budget, indent=2, allow_nan=False) + "\n"
)
print(json.dumps(ending_budget, indent=2))
if ENDING_UNITS_REMAINING is not None and ENDING_UNITS_REMAINING <= RESERVE_UNITS:
    print("The 50-unit reserve has been reached. Export and disconnect; do not start another run.")

## Download the reports and checkpoint

Run this after success or failure, before disconnecting. The ZIP recursively includes available
checkpoints, frozen configuration, reports, metadata and logs from this run folder, but no database.
Confirm the downloaded ZIP contains `checkpoints/training.pt` when training produced a checkpoint.
Keep it for resuming and later fixed-checkpoint evaluation. Then **disconnect and delete the
runtime**. Do not leave an idle GPU connected.

In [ ]:
from google.colab import files

archive = shutil.make_archive("/content/trgnn-training-run-" + run_id, "zip", root_dir=str(RUN_DIR))
files.download(archive)